# Baseline 학습 - fold 0

| 항목 | 설정 | 근거 |
|---|---|---|
| 인코더 | DINOv2-S/14 (22M) | Base 는 노이즈 바닥 대비 null (735154) |
| 입력 | 6칸 x 9슬라이스 x 224px | 물리 크롭 130mm, 0.5804 mm/px |
| 슬롯 | plan C (평면 통합) | T1 보유 92.6% vs 시상면 T1 40.4% |
| 타깃 | stevenleehans v4_blend | gold 58건 대비 0.8927 (docs/labels.md) |
| 손실 | soft BCE + 신뢰도 가중 | 미언급 셀(0.25)은 가중 0.3 |
| fold | 1 fold 만 | 5-fold 앙상블과 차이 없음 (740375) |

**푸시 전 DEBUG=True 로 스모크 테스트를 통과시킬 것**


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
for _c in ['/kaggle/input/datasets/hyejinkwak/knee-src', '/kaggle/input/knee-src']:
    if os.path.isdir(_c):
        sys.path.insert(0, _c); break

import torch, numpy as np, pandas as pd
import paths, config, dicom_io, dataset as ds, labels as lb, model as M, train as T

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    cc = torch.cuda.get_device_capability(0)
    print('compute capability', cc)
    assert cc >= (7, 0), ('P100(sm_60)은 사전설치 PyTorch 와 호환되지 않는다. '
                          'kernel-metadata.json 에 machine_shape=NvidiaTeslaT4 를 넣을 것')
config.verify_labels()
print(paths.describe())


## 설정

`DEBUG=True` 면 스터디 64건, 1 epoch 만 돈다


In [ ]:
DEBUG = True      # <- 실제 학습 시 False

CACHE = paths.find_input('knee-cache-build')
print('cache:', CACHE)

cfg = T.TrainConfig(
    fold=0, plan='C', epochs=8, batch_size=8, accum=2,
    lr=1e-4, num_workers=4, debug=DEBUG, out_dir='/kaggle/working')

WEIGHTS = None
try:
    wd = paths.find_input('knee-dinov2-weights')
    hits = sorted(Path(wd).glob('**/*.pth'))
    assert hits, f'{wd} 에 .pth 가 없다'
    WEIGHTS = str(hits[0])
    wmeta = json.loads((Path(wd) / 'weights_meta.json').read_text())
    print('offline weights:', WEIGHTS)
    print('  ', wmeta)
    assert wmeta['img_size'] == 224, 'img_size 가 캐시와 다르다'
except FileNotFoundError:
    print('[!] knee-dinov2-weights 없음 - pretrained 다운로드를 시도한다')

mcfg = M.ModelConfig(pretrained=WEIGHTS is None, weights_path=WEIGHTS)
print(cfg)
print(mcfg)


## 캐시 확인

슬롯 수가 코드와 다르면 즉시 멈춘다


In [ ]:
cache = ds.open_cache(CACHE)
print('shards', len(cache.arrays), '| studies', len(cache.index))
print('slots', cache.slots)

pick, mask = ds.build_slot_map(cache, cfg.plan)
cov = pd.DataFrame(mask, columns=[f'slot{i}' for i in range(mask.shape[1])])
print(f'\nplan {cfg.plan} 칸별 채움률')
print(cov.mean().round(3).to_string())
print(f'평균 {cov.mean().mean():.3f}')


## 학습


In [ ]:
report = T.run(str(CACHE), cfg, mcfg)
print()
print(json.dumps(report['best'], indent=2))


## 결과


In [ ]:
h = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_label'}
                  for r in report['history']])
print(h.to_string(index=False))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(h.epoch, h.loss, marker='o'); ax[0].set_title('train loss', fontsize=9)
ax[1].plot(h.epoch, h.val_auc, marker='o', label='valid (report labels)')
if h.gold_auc.notna().any():
    ax[1].plot(h.epoch, h.gold_auc, marker='s', label='gold panel')
ax[1].set_title('macro AUC', fontsize=9); ax[1].legend(fontsize=8)
for a in ax: a.set_xlabel('epoch')
plt.tight_layout()
plt.savefig('/kaggle/working/train_curve.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
best = max(report['history'], key=lambda r: r['val_auc'])
per = pd.Series(best['per_label']).sort_values()
print('레이블별 AUC (best epoch)')
print(per.round(4).to_string())
print(f'\nmacro {best["val_auc"]:.4f}')
print()
print('검증 타깃이 판독문 유래라는 점을 기억할 것')
print('이 수치는 판독문을 얼마나 잘 맞추나이지 무릎을 얼마나 잘 보나가 아니다')
print(f'\n총 소요 {(time.time()-T0)/60:.1f}분')
print('output:', sorted(os.listdir('/kaggle/working')))
